# 84. 共享单车需求与运力调度

<!-- module-learning-arc:start -->
> **综合项目 模块主线｜第 3 / 4 步：从历史变化支持资源规划**
>
> **持续应用背景：** 进入数据分析决策实验室：连续处理客户价值、物流履约、供需调度和营销资源四类问题，训练从业务问题到行动建议的迁移能力。
>
> **承接上一阶段：** Olist电商物流履约分析  →  **本章任务：** 共享单车需求与运力调度  →  **下一步：** 银行客户营销转化分析
>
> **大作业连接：** 本章练习将成为《跨模块业务决策项目》的一部分，最终需要把前四个项目形成的方法迁移为项目提案、最短充分证据链和决策备忘录。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：共享单车平台的运营每天都要回答一个问题：哪些时段、哪些天气条件下会有多少人骑车？提前知道某时段需求高，就能在高峰前把车调到位，避免用户找不到车；预测到雨天冷清，就能少派补车、节省成本。本项目用 UCI Bike Sharing 的小时级租赁记录（2011–2012 两年共 17,379 条）来解决这个调度问题，先从数据字典认识每一列的用途，再逐步走到补车建议。



## 本章目标

学完本章，你将能够：

- **理解**：理解「共享单车需求与运力调度」的核心概念、适用场景与关键口径。
- **操作**：能按本章步骤写出可复现的实现，并读懂输出/结果。
- **迁移**：能用本章方法处理一份新数据，独立完成同类任务并给出结论。


## 84.1 数据字典

共享单车平台的运营每天都要回答一个问题：哪些时段、哪些天气条件下会有多少人骑车？提前知道某时段需求高，就能在高峰前把车调到位，避免用户找不到车；预测到雨天冷清，就能少派补车、节省成本。本项目用 UCI Bike Sharing 的小时级租赁记录（2011–2012 两年共 17,379 条）来解决这个调度问题，先从数据字典认识每一列的用途，再逐步走到补车建议。


| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| dteday/hr | 日期/小时 | 需求时间 |
| workingday/holiday | 工作日/节假日 | 日历变量 |
| weathersit | 天气等级 | 1好至4差 |
| temp/atemp/hum/windspeed | 归一化气象指标 | 连续变量 |
| casual/registered | 临时/注册用户数 | cnt 的组成，不可作预测特征 |
| cnt | 总租赁量 | 预测目标 |

## 84.2 数据质量检查清单

- instant 是否唯一
- 日期小时是否重复
- cnt 是否等于 casual+registered
- 时间是否连续及是否存在缺口
- 归一化气象字段范围


## 84.3 项目任务

1. 结构审计与时间索引
2. 分析小时和工作日需求
3. 比较用户类型
4. 分析天气条件
5. 用时间切分训练需求模型
6. 输出补车建议与限制


## 84.4 项目阶段速查

先看每个阶段要做什么、留下什么证据，再按任务顺序运行项目代码。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 1. 加载与结构审计 | `pd.read_csv()`、`pd.to_timedelta()`、`df.sort_values()`、`df.timestamp.min()` | 验证目标构成关系，并建立真正的时间顺序。 | instant 是否唯一 |
| 2. 通勤峰谷与用户结构 | `df.groupby()`、`plt.subplots()`、`ax.set_ylabel()`、`plt.tight_layout()` | 注册用户通常体现通勤规律，临时用户更受休闲场景影响；分开看比总量更有运营价值。 | 日期小时是否重复 |
| 3. 天气与运力风险 | `df.groupby()`、`x.quantile()`、`x.sum()`、`weather.round()` | 天气不是随机分配的，比较用于排班情景而非因果断言。 | cnt 是否等于 casual+registered |
| 4. 时间切分需求预测 | `model.fit()`、`model.predict()`、`np.repeat()`、`train.cnt.tail()` | 按时间保留最后 20% 做测试，不随机打乱未来；明确排除 casual 和 registered 两个目标组成字段。 | 时间是否连续及是否存在缺口 |
| 5. 调度建议 | `cnt.idxmax()`、`loc[1]`、`loc[0]` | 预测必须进入库存、站点容量和调度成本体系后才能落地。 | 归一化气象字段范围 |


## 84.5 项目交付物

- 一份可复现的分析 Notebook
- 清洗规则与关键指标表
- 至少一张支持结论的图表
- 结论、限制和下一步建议

## 84.6 阶段检查点

- [ ] 问题和数据字典完成
- [ ] 质量检查和清洗记录完成
- [ ] 核心指标或图表完成
- [ ] 结论与限制完成

## 84.7 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 84.8 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 84.9 加载与结构审计

验证目标构成关系，并建立真正的时间顺序。


<!-- math-foundation:chapter-84 -->
### 数学推导｜有限资源下的需求覆盖率

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜计算全部需求。** $D=\sum_id_i$。

**第 2 步｜给定行动集合 $S$。** 可被覆盖的需求是 $D_S=\sum_{i\in S}d_i$。

**第 3 步｜形成 0 到 1 的指标。** $coverage(S)=D_S/D$。再加入一个时段 $j$ 的边际提升为

$$
\Delta coverage_j=coverage(S\cup\{j\})-coverage(S)=\frac{d_j}{D}
$$

在成本相同且需求互不重叠时，按 $d_j$ 从大到小选择会最快提高覆盖率。

**把上面的关系收束为本章计算式：**

$$
coverage(S)=\frac{\sum_{i\in S}d_i}{\sum_i d_i}
$$

**符号解释：** $S$ 是被选中的高峰时段集合，$d_i$ 是该时段需求。

**代码对应：** 按需求或风险排序选择时段，并比较不同阈值下的覆盖率与行动量。

**使用边界：** 历史覆盖率不是未来保证；还需考虑站点容量、成本和需求漂移。


In [ ]:
import os

import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
import pandas as pd
import numpy as np

# 中文字体支持：自动选用可用的中文字体，避免图表中文显示为方框
if os.path.exists("/tmp/NotoSansSC-Regular.otf"):
    fm.fontManager.addfont("/tmp/NotoSansSC-Regular.otf")
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("/datasets/bike_sharing_hour.csv", parse_dates=["dteday"])
df["timestamp"] = df.dteday + pd.to_timedelta(df.hr, unit="h")
df = df.sort_values("timestamp")
print("形状:", df.shape, " 时间:", df.timestamp.min(), "至", df.timestamp.max())
print(
    "时间重复:",
    df.timestamp.duplicated().sum(),
    " 目标构成错误:",
    (df.cnt != df.casual + df.registered).sum(),
)
print(df[["temp", "atemp", "hum", "windspeed", "cnt"]].describe().round(2))


**练一练：统计不同天气等级的样本量**

加载好的 `df` 记录了每个小时的总租赁量 `cnt` 和天气等级 `weathersit`（1 最好、4 最差）。
请在下方补全代码：按 `weathersit` 分组，统计每个天气等级下有多少个小时（行数），以及对应的平均总租赁量 `cnt`，结果保存到变量 `weather_count`。

提示：`df.groupby(<分组列名>)` 的括号里要填分组依据的列名，后面用 `.agg()` 汇总。


In [ ]:
# 请在下方填写代码
# 补全下方 groupby 括号里的分组列名（天气等级那一列），
# 统计每个天气等级对应的小时数（行数）和平均租赁量 cnt。


In [ ]:
# 参考答案：把上一格 groupby 括号中的横线补成字符串 "weathersit"
weather_count = df.groupby("weathersit").agg(
    hours=("cnt", "size"),
    mean_demand=("cnt", "mean"),
)


## 84.10 通勤峰谷与用户结构

注册用户通常体现通勤规律，临时用户更受休闲场景影响；分开看比总量更有运营价值。


In [ ]:
# 中文字体支持：避免图表中文显示为方框
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

profile = df.groupby(["workingday", "hr"])[
    ["casual", "registered", "cnt"]
].mean()
print("工作日需求最高小时:\n", profile.loc[1].nlargest(5, "cnt").round(1))
print("非工作日需求最高小时:\n", profile.loc[0].nlargest(5, "cnt").round(1))
fig, ax = plt.subplots(figsize=(9, 4))
profile.loc[1, ["casual", "registered"]].plot(ax=ax, title="工作日：临时与注册用户小时需求")
ax.set_ylabel("平均租赁量")
plt.tight_layout()
plt.show()


## 84.11 天气与运力风险

天气不是随机分配的，比较用于排班情景而非因果断言。


In [ ]:
weather = df.groupby("weathersit").agg(
    hours=("cnt", "size"),
    mean_demand=("cnt", "mean"),
    p90_demand=("cnt", lambda x: x.quantile(0.9)),
    registered_share=(
        "registered",
        lambda x: x.sum() / df.loc[x.index, "cnt"].sum(),
    ),
)
print(weather.round(2))
print(
    "各场景P90:",
    df.groupby(["workingday", "weathersit"])
    .cnt.quantile(0.9)
    .round(0)
    .to_dict(),
)


## 84.12 时间切分需求预测

按时间保留最后 20% 做测试，不随机打乱未来；明确排除 casual 和 registered 两个目标组成字段。


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

features = [
    "season",
    "yr",
    "mnth",
    "hr",
    "holiday",
    "weekday",
    "workingday",
    "weathersit",
    "temp",
    "atemp",
    "hum",
    "windspeed",
]
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]
cat = ["season", "mnth", "hr", "weekday", "weathersit"]
num = [c for c in features if c not in cat]
prep = ColumnTransformer(
    [
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            cat,
        ),
        ("num", "passthrough", num),
    ]
)
model = Pipeline(
    [
        ("prep", prep),
        (
            "model",
            HistGradientBoostingRegressor(max_iter=180, random_state=74),
        ),
    ]
)
model.fit(train[features], train.cnt)
pred = model.predict(test[features])
baseline = np.repeat(train.cnt.tail(24 * 28).mean(), len(test))
print("时间测试区间:", test.timestamp.min(), "至", test.timestamp.max())
print(
    "模型 MAE:",
    round(mean_absolute_error(test.cnt, pred), 1),
    " 均值基线 MAE:",
    round(mean_absolute_error(test.cnt, baseline), 1),
)


## 84.13 调度建议

预测必须进入库存、站点容量和调度成本体系后才能落地。


In [ ]:
work_peak = profile.loc[1].cnt.idxmax()
off_peak = profile.loc[0].cnt.idxmax()
print(f"1. 工作日全网峰值约在 {work_peak}:00，非工作日约在 {off_peak}:00，补车应在峰值前完成。")
print("2. 以工作日×天气等级的P90作为初始运力情景，并用滚动时间窗回测。")
print("3. 当前数据没有站点库存、OD流向和调度成本，只能做全网需求预测，不能直接生成调度路线。")


## 84.14 本章实训：从原始记录到质量报告

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03", "C04"],
        "amount": [120, 80, 80, None, -20],
    }
)
quality = pd.Series(
    {
        "原始行数": len(raw),
        "重复行数": raw.duplicated().sum(),
        "缺失金额": raw["amount"].isna().sum(),
        "非正金额": (raw["amount"] <= 0).sum(),
    }
)
print(quality.to_string())


### 84.14.1 第一个结果怎么读

项目的第一步不是急着画图或建模，而是量化问题规模。质量报告要能回答：问题有多少、影响哪些字段、下一步如何处理。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
clean = raw.drop_duplicates().copy()
clean["amount_valid"] = clean["amount"].where(clean["amount"] > 0)
summary = clean.groupby("customer_id", as_index=False)["amount_valid"].sum(
    min_count=1
)
print("清洗后行数：", len(clean))
print("有效客户数：", summary["amount_valid"].notna().sum())
print(summary)


### 84.14.2 第二个结果怎么读

第二个实验把质量问题转成可追踪的清洗结果。请同时记录删除、保留和缺失处理规则，不能只报告最后的数字。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 84.15 错误恢复：重复主键和缺失值怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03"],
        "amount": [120, 80, None, -20],
    }
)
duplicate_keys = raw["customer_id"].duplicated(keep=False)
invalid_amount = raw["amount"].isna() | raw["amount"].le(0)
print("重复主键行：")
print(raw[duplicate_keys])
print("金额异常行：")
print(raw[invalid_amount])
print("先标记问题，再决定保留、合并或回查。")


### 84.15.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

项目中不能把异常行静默删除。先输出问题记录和数量，再把处理规则写进项目结论。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 84.16 易错点提醒

**易错点 1**：weathersit 是类别编码不是数值，直接当连续值算均值没有意义；需要时用哑变量或映射成"天气状况"标签。

**易错点 2**：hr 是 0-23 的小时编号，画图或分组时按类别处理，不要当连续变量求平均。

**易错点 3**：随机切分训练/测试会泄漏时间顺序，需求分析应按时段切分（如按日期先后）。

**易错点 4**：cnt 包含 casual 与 registered 之和，分析时明确用哪个口径，避免重复计数。

**易错点 5**：极端天气日（恶劣天气、节假日）的 cnt 异常低，做均值或趋势前先看分布，别让异常值拉偏结论。


## 84.17 结论与表达

- 用户类型拆分揭示通勤与休闲需求差异。
- 预测 cnt 时使用 casual/registered 会造成目标泄漏。
- 时间切分比随机切分更接近未来预测。
- 全网需求还不是站点级调度方案。


## 84.18 项目验收清单

- 验证 cnt 构成关系
- 能识别工作日峰值
- 模型排除泄漏字段
- 使用时间测试集并与基线比较

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 84.19 小结

使用 UCI Bike Sharing 17,379 条小时级租赁记录，分析通勤峰谷、天气冲击和注册/临时用户差异，并建立无泄漏需求预测基线。


### 84.19.1 你已经完成

- 理解小时级时间序列结构
- 区分临时与注册用户需求
- 识别工作日和天气下的峰谷
- 避免使用 casual/registered 预测 cnt 的目标泄漏
- 用时间切分评估需求预测


### 84.19.2 质量与结论提醒

- instant 是否唯一
- 日期小时是否重复
- cnt 是否等于 casual+registered
- 用户类型拆分揭示通勤与休闲需求差异。
- 预测 cnt 时使用 casual/registered 会造成目标泄漏。
- 时间切分比随机切分更接近未来预测。
- 全网需求还不是站点级调度方案。


### 84.19.3 项目交付检查

- [ ] 验证 cnt 构成关系
- [ ] 能识别工作日峰值
- [ ] 模型排除泄漏字段
- [ ] 使用时间测试集并与基线比较


### 84.19.4 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
